## Обучение основных моделей sklearn 

In [2]:
import pandas as pd
import pickle
import warnings
import optuna
import os
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import fbeta_score

from sklearn.linear_model import LogisticRegressionCV, RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC

Здесь будут обучены модели на варианте текста без стемминга

In [3]:
full_df = pd.read_csv(fr'DataBases\prepared_datasets\full_dataset_emotions_text.csv').dropna()
full_df

,text,emotion,full_prep_text,no_stem_text
0,carefully word blog posts amount criticism hea...,0,care word blog post amount critic hear place c...,carefully word blog posts amount criticism hea...
1,cannot remember little mermaid feeling carefre...,1,rememb littl mermaid feel carefre beauti life ...,remember little mermaid feeling carefree beaut...
2,not feeling super well turns cold knocked next...,1,feel super well turn cold knock next three wee...,feeling super well turns cold knocked next thr...
3,feel honored part group amazing talents,1,feel honor part group amaz talent,feel honored part group amazing talents
4,think helping also began feel pretty lonely lo...,0,think help also began feel pretti lone lot peo...,think helping also began feel pretty lonely lo...
...,...,...,...,...
282817,feel honored motivated share world life changi...,1,feel honor motiv share world life chang gift a...,feel honored motivated share world life changi...
282818,feel like gloaty really delighted,1,feel like gloati realli delight,feel like gloaty really delighted
282819,feel little energetic one day next several day...,1,feel littl energet one day next sever day hard...,feel little energetic one day next several day...
282820,feel work experience fell although fantastic o...,1,feel work experi fell although fantast opportu...,feel work experience fell although fantastic o...


In [4]:
x_train, x_test, y_train, y_test = train_test_split(full_df.no_stem_text, full_df.emotion, test_size=0.15, random_state=42, stratify=full_df.emotion)
x_train.shape, x_test.shape

((240397,), (42424,))

In [5]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 282821 entries, 0 to 282821
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   text            282821 non-null  object
 1   emotion         282821 non-null  int64 
 2   full_prep_text  282821 non-null  object
 3   no_stem_text    282821 non-null  object
dtypes: int64(1), object(3)
memory usage: 10.8+ MB


In [5]:
vectorizer = TfidfVectorizer()
x_train_features = vectorizer.fit_transform(x_train)
x_test_features = vectorizer.transform(x_test)

In [6]:
with open('vectorizers/vect_no_stem_text.pickle', 'wb') as handle:
    pickle.dump(vectorizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [6]:
with open('vectorizers/vect_no_stem_text.pickle', 'rb') as handle:
    vectorizer_no_stem = pickle.load(handle)

In [ ]:
models_to_fit = [
    'LogisticRegressionCV',
    'RidgeClassifierCV',
    'RandomForestClassifier',
    'CatBoostClassifier',
    'LGBMClassifier',
    'XGBClassifier',
    'MultinomialNB',
    'SVC'
]

In [8]:
from sklearn.metrics import precision_score, recall_score

In [9]:
def custom_loss(y_test, y_pred):
    return 0.6 * recall_score(y_test, y_pred) + 0.4 * precision_score(y_test, y_pred)

In [8]:
def objective(trial, model_name):
    if model_name == 'CatBoostClassifier':
        model = CatBoostClassifier(
            iterations=trial.suggest_int('iterations', 1000, 2500),
            depth=trial.suggest_int('depth', 3, 10),
            learning_rate=trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
            early_stopping_rounds=20,  # борьба против переобучения (через 20 итераций неизменяемого лосса обучение останавливается)
            metric_period = 200,
            task_type="GPU",
            used_ram_limit='18gb'
        )

    elif model_name == 'LogisticRegressionCV':
        model = LogisticRegressionCV(
            Cs=trial.suggest_int('Cs', 1, 15),
            cv=trial.suggest_int('cv', 3, 12),
            max_iter=trial.suggest_int('max_iter', 500, 2500),
            scoring='recall',
            penalty='l2',
            n_jobs=-1,
        )

    elif model_name == 'RidgeClassifierCV':
        model = RidgeClassifierCV(
            alphas=[trial.suggest_float('alpha', 0.1, 15)],
            cv=trial.suggest_int('cv', 3, 12)
        )

    elif model_name == 'RandomForestClassifier':
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 800),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            n_jobs=-1,
        )

    elif model_name == 'LGBMClassifier':
        model = LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 200, 1000),
            learning_rate=trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 8, 256),
            subsample=trial.suggest_float('subsample', 0.5, 1),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1),
            n_jobs=-1,
            force_col_wise=True,
            verbosity=-1
        )

    elif model_name == 'XGBClassifier':
        model = XGBClassifier(
            n_estimators=trial.suggest_int('n_estimators', 200, 1000),
            learning_rate=trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            max_depth=trial.suggest_int('max_depth', 3, 10),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            use_label_encoder=False,
            eval_metric='logloss',
            n_jobs=-1,
            verbosity=0
        )

    elif model_name == 'MultinomialNB':
        model = MultinomialNB(
            alpha=trial.suggest_float('alpha', 1e-3, 10.0, log=True)
        )

    elif model_name == 'SVC':
        model = SVC(
            C=trial.suggest_float('C', 1e-3, 10.0, log=True),
            kernel=trial.suggest_categorical('kernel', ['linear', 'rbf']),
            gamma='scale',
            probability=False,
            verbose=True
        )

    pipeline = Pipeline([
        ('tfidf', vectorizer_no_stem),
        ('model', model)
    ])

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    # метрика fbeta с бОльшим весом для recall
    return fbeta_score(y_test, y_pred, beta=2, average='weighted', zero_division=0)
    #return custom_loss(y_test, y_pred)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)  # чтобы не выводить кучу лишнего (только информация об обучении моделей)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for model_name in models_to_fit:
        print(f"\nОптимизация модели: {model_name}")

        study = optuna.create_study(direction="maximize", study_name=f"{model_name}_optimization")
        study.optimize(lambda trial: objective(trial, model_name), n_trials=50)

        best_params = study.best_params
        best_score = study.best_value

        print(f"Наилучшие результаты для {model_name}")
        print(f"Fbeta (β=2): {best_score:.4f}")
        print("Лучшие параметры:", best_params)
        print("-" * 50)

        with open(f'models_best_params/{model_name}.pickle', 'wb') as handle:
            pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

[I 2025-11-09 21:46:50,240] A new study created in memory with name: CatBoostClassifier_optimization



Оптимизация модели: CatBoostClassifier


: 

Catboost обучался слишком долго без логов, поэтому пришлось прервать процесс обучения. Далее возобновляем с логами с него же

In [ ]:
models_to_fit = [
    'CatBoostClassifier',
    'LGBMClassifier',
    'XGBClassifier',
    'MultinomialNB',
    'SVC'
]

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)  # чтобы не выводить кучу лишнего (только информация об обучении моделей)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for model_name in models_to_fit:
        print(f"\nОптимизация модели: {model_name}")

        study = optuna.create_study(direction="maximize", study_name=f"{model_name}_optimization")
        study.optimize(lambda trial: objective(trial, model_name), n_trials=50)

        best_params = study.best_params
        best_score = study.best_value

        print(f"Наилучшие результаты для {model_name}")
        print(f"Fbeta (β=2): {best_score:.4f}")
        print("Лучшие параметры:", best_params)
        print("-" * 50)

        with open(f'models_best_params/{model_name}.pickle', 'wb') as handle:
            pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

[I 2025-11-09 21:49:58,816] A new study created in memory with name: CatBoostClassifier_optimization



Оптимизация модели: CatBoostClassifier
0:	learn: 0.6923431	total: 262ms	remaining: 4m 21s
200:	learn: 0.5723266	total: 15.4s	remaining: 1m 1s
400:	learn: 0.4892776	total: 32.9s	remaining: 49.2s
600:	learn: 0.4261387	total: 54.6s	remaining: 36.3s
800:	learn: 0.3774039	total: 1m 17s	remaining: 19.3s
1000:	learn: 0.3389998	total: 1m 39s	remaining: 0us


[I 2025-11-09 21:52:18,331] Trial 0 finished with value: 0.9222582046626243 and parameters: {'iterations': 1001, 'depth': 3, 'learning_rate': 0.027224117731881898, 'l2_leaf_reg': 0.0013166056570913871}. Best is trial 0 with value: 0.9222582046626243.


0:	learn: 0.6909952	total: 259ms	remaining: 8m 19s
200:	learn: 0.4279611	total: 26.8s	remaining: 3m 50s
400:	learn: 0.3093561	total: 49.9s	remaining: 3m 9s
600:	learn: 0.2451212	total: 1m 11s	remaining: 2m 37s
800:	learn: 0.2069309	total: 1m 31s	remaining: 2m 9s
1000:	learn: 0.1819264	total: 1m 53s	remaining: 1m 44s
1200:	learn: 0.1639273	total: 2m 16s	remaining: 1m 22s
1400:	learn: 0.1506569	total: 2m 43s	remaining: 1m 1s
1600:	learn: 0.1406292	total: 3m 4s	remaining: 37.8s
1800:	learn: 0.1332576	total: 3m 26s	remaining: 14.6s
1927:	learn: 0.1296328	total: 3m 40s	remaining: 0us


[I 2025-11-09 21:56:57,830] Trial 1 finished with value: 0.9552847083663165 and parameters: {'iterations': 1928, 'depth': 5, 'learning_rate': 0.0473911884923361, 'l2_leaf_reg': 0.008331434401416012}. Best is trial 1 with value: 0.9552847083663165.


0:	learn: 0.6871588	total: 434ms	remaining: 7m 56s
200:	learn: 0.2432748	total: 33.4s	remaining: 2m 29s
400:	learn: 0.1633413	total: 1m 9s	remaining: 2m 1s
600:	learn: 0.1345865	total: 1m 40s	remaining: 1m 23s
800:	learn: 0.1242712	total: 2m 7s	remaining: 47.6s
1000:	learn: 0.1194855	total: 2m 36s	remaining: 15.3s
1098:	learn: 0.1174670	total: 2m 50s	remaining: 0us


[I 2025-11-09 22:00:57,845] Trial 2 finished with value: 0.9563712446155505 and parameters: {'iterations': 1099, 'depth': 7, 'learning_rate': 0.10003700274837302, 'l2_leaf_reg': 2.0936194372133694}. Best is trial 2 with value: 0.9563712446155505.


0:	learn: 0.6911069	total: 383ms	remaining: 9m 33s
200:	learn: 0.4378396	total: 19.7s	remaining: 2m 7s
400:	learn: 0.3187645	total: 39.8s	remaining: 1m 48s
600:	learn: 0.2536833	total: 1m	remaining: 1m 29s
800:	learn: 0.2140552	total: 1m 20s	remaining: 1m 10s
1000:	learn: 0.1879093	total: 1m 40s	remaining: 50s
1200:	learn: 0.1698682	total: 2m 1s	remaining: 30s
1400:	learn: 0.1561138	total: 2m 22s	remaining: 9.77s
1496:	learn: 0.1508931	total: 2m 32s	remaining: 0us


[I 2025-11-09 22:04:33,511] Trial 3 finished with value: 0.953655435366311 and parameters: {'iterations': 1497, 'depth': 5, 'learning_rate': 0.04501886487689178, 'l2_leaf_reg': 0.2778136862971692}. Best is trial 2 with value: 0.9563712446155505.


: 

При обучении Catboost очень быстро отмирает ядро (за счет очень большого использования оперативной памяти. процессора и gpu). Оставим модель, которая дает лучшие результаты из проведенных попыток

In [9]:
with open(f'models_best_weights/CatBoostClassifier.pickle', 'wb') as handle:
    pickle.dump({'iterations': 1099, 'depth': 7, 'learning_rate': 0.10003700274837302, 'l2_leaf_reg': 2.0936194372133694}, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
models_to_fit = [
    'LGBMClassifier',
    'XGBClassifier',
    'MultinomialNB',
    'SVC'
]

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)  # чтобы не выводить кучу лишнего (только информация об обучении моделей)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for model_name in models_to_fit:
        print(f"\nОптимизация модели: {model_name}")

        study = optuna.create_study(direction="maximize", study_name=f"{model_name}_optimization")
        study.optimize(lambda trial: objective(trial, model_name), n_trials=50)

        best_params = study.best_params
        best_score = study.best_value

        print(f"Наилучшие результаты для {model_name}")
        print(f"Fbeta (β=2): {best_score:.4f}")
        print("Лучшие параметры:", best_params)
        print("-" * 50)

        with open(f'models_best_params/{model_name}.pickle', 'wb') as handle:
            pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

[I 2025-11-09 22:51:14,425] A new study created in memory with name: LGBMClassifier_optimization



Оптимизация модели: LGBMClassifier


[I 2025-11-09 22:51:33,119] Trial 0 finished with value: 0.9617906524251281 and parameters: {'n_estimators': 726, 'learning_rate': 0.2669115835149503, 'num_leaves': 16, 'subsample': 0.7069642755713801, 'colsample_bytree': 0.7385280898887101}. Best is trial 0 with value: 0.9617906524251281.
[I 2025-11-09 22:53:40,913] Trial 1 finished with value: 0.944499087019166 and parameters: {'n_estimators': 814, 'learning_rate': 0.0015060440239239772, 'num_leaves': 193, 'subsample': 0.5898554520160434, 'colsample_bytree': 0.9189416283615139}. Best is trial 0 with value: 0.9617906524251281.
[I 2025-11-09 22:55:26,356] Trial 2 finished with value: 0.9530101457023789 and parameters: {'n_estimators': 589, 'learning_rate': 0.0035246042377633227, 'num_leaves': 224, 'subsample': 0.7708695537317978, 'colsample_bytree': 0.9204640766079366}. Best is trial 0 with value: 0.9617906524251281.
[I 2025-11-09 22:56:26,140] Trial 3 finished with value: 0.9313820271715186 and parameters: {'n_estimators': 404, 'learn

Наилучшие результаты для LGBMClassifier
Fbeta (β=2): 0.9640
Лучшие параметры: {'n_estimators': 921, 'learning_rate': 0.01665933885210038, 'num_leaves': 166, 'subsample': 0.8112788681330841, 'colsample_bytree': 0.9239101823925396}
--------------------------------------------------

Оптимизация модели: XGBClassifier


[I 2025-11-10 00:52:07,068] Trial 0 finished with value: 0.5738907170516538 and parameters: {'n_estimators': 626, 'learning_rate': 0.001595072234467645, 'max_depth': 4, 'subsample': 0.7168819494560071, 'colsample_bytree': 0.6738405949877199}. Best is trial 0 with value: 0.5738907170516538.
[I 2025-11-10 00:52:58,204] Trial 1 finished with value: 0.5902920528638547 and parameters: {'n_estimators': 289, 'learning_rate': 0.002524593171408986, 'max_depth': 6, 'subsample': 0.6938935980241295, 'colsample_bytree': 0.8258869312216746}. Best is trial 1 with value: 0.5902920528638547.
[I 2025-11-10 00:54:11,351] Trial 2 finished with value: 0.9172154801190873 and parameters: {'n_estimators': 227, 'learning_rate': 0.058559535567413855, 'max_depth': 10, 'subsample': 0.8067586590732942, 'colsample_bytree': 0.8276574318024372}. Best is trial 2 with value: 0.9172154801190873.
[I 2025-11-10 00:56:02,673] Trial 3 finished with value: 0.71782228196764 and parameters: {'n_estimators': 602, 'learning_rate

Наилучшие результаты для XGBClassifier
Fbeta (β=2): 0.9621
Лучшие параметры: {'n_estimators': 925, 'learning_rate': 0.24134341324627948, 'max_depth': 10, 'subsample': 0.7912325993179473, 'colsample_bytree': 0.926630684907128}
--------------------------------------------------

Оптимизация модели: MultinomialNB


[I 2025-11-10 02:36:47,219] Trial 0 finished with value: 0.937715703410627 and parameters: {'alpha': 3.4455973950689143}. Best is trial 0 with value: 0.937715703410627.
[I 2025-11-10 02:36:48,813] Trial 1 finished with value: 0.9291467424847037 and parameters: {'alpha': 0.33182101716002327}. Best is trial 0 with value: 0.937715703410627.
[I 2025-11-10 02:36:50,435] Trial 2 finished with value: 0.9023906737228327 and parameters: {'alpha': 0.0031477339941778453}. Best is trial 0 with value: 0.937715703410627.
[I 2025-11-10 02:36:52,083] Trial 3 finished with value: 0.9318082155911728 and parameters: {'alpha': 0.4835819882948738}. Best is trial 0 with value: 0.937715703410627.
[I 2025-11-10 02:36:53,657] Trial 4 finished with value: 0.9371944320353923 and parameters: {'alpha': 4.54780367714309}. Best is trial 0 with value: 0.937715703410627.
[I 2025-11-10 02:36:55,240] Trial 5 finished with value: 0.9024606271341732 and parameters: {'alpha': 0.0037337861013625718}. Best is trial 0 with va

Наилучшие результаты для MultinomialNB
Fbeta (β=2): 0.9381
Лучшие параметры: {'alpha': 2.945061227669654}
--------------------------------------------------

Оптимизация модели: GaussianNB


[W 2025-11-10 02:38:04,718] Trial 0 failed with parameters: {'var_smoothing': 2.7779869271615698e-12} because of the following error: TypeError("Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.").
Traceback (most recent call last):
  File "c:\interpreter\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_59180\3834890825.py", line 9, in <lambda>
    study.optimize(lambda trial: objective(trial, model_name), n_trials=50)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_59180\3483072427.py", line 87, in objective
    pipeline.fit(x_train, y_train)
  File "c:\interpreter\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.

In [18]:
models_to_fit = [
    'SVC'
]

In [ ]:
def objective(trial, model_name):
    if model_name == 'SVC':
        model = SVC(
            C=trial.suggest_float('C', 1e-3, 10.0, log=True),
            kernel=trial.suggest_categorical('kernel', ['linear', 'rbf']),
            gamma='scale',
            probability=False,
            verbose=True
        )


    pipeline = Pipeline([
        ('tfidf', vectorizer_no_stem),
        ('model', model)
    ])

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    # Сохраняем параметры текущей итерации, если это лучший результат
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    # Сохраняем параметры текущего trial
    current_params = trial.params
    with open(f'models_best_params/{model_name}_current.pickle', 'wb') as handle:
        pickle.dump(current_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

    return fbeta_score(y_test, y_pred, beta=2, average='weighted', zero_division=0)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)  # чтобы не выводить кучу лишнего (только информация об обучении моделей)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for model_name in models_to_fit:
        print(f"\nОптимизация модели: {model_name}")

        study = optuna.create_study(direction="maximize", study_name=f"{model_name}_optimization")
        study.optimize(lambda trial: objective(trial, model_name), n_trials=50)

        best_params = study.best_params
        best_score = study.best_value

        print(f"Наилучшие результаты для {model_name}")
        print(f"Fbeta (β=2): {best_score:.4f}")
        print("Лучшие параметры:", best_params)
        print("-" * 50)

        with open(f'models_best_params/{model_name}.pickle', 'wb') as handle:
            pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

[I 2025-11-10 23:21:01,391] A new study created in memory with name: SVC_optimization



Оптимизация модели: SVC
[LibSVM]

[I 2025-11-11 01:03:36,339] Trial 0 finished with value: 0.960539775317096 and parameters: {'C': 4.60365884498761, 'kernel': 'rbf'}. Best is trial 0 with value: 0.960539775317096.


[LibSVM]

[I 2025-11-11 02:44:17,332] Trial 1 finished with value: 0.9605399115073129 and parameters: {'C': 8.17929352943562, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9605399115073129.


...

[LibSVM]

[I 2025-11-11 07:54:23,536] Trial 6 finished with value: 0.9546709635293422 and parameters: {'C': 6.63707720483188, 'kernel': 'linear'}. Best is trial 1 with value: 0.9605399115073129.


[LibSVM]

[I 2025-11-11 08:46:37,527] Trial 7 finished with value: 0.9570294117110245 and parameters: {'C': 0.25821660188064866, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9605399115073129.


[LibSVM]

Модели SVC ввиду своей высокой вычислительной сложности и большого объема датасета обучаются очень долго. Скорее всего, точно так же долго они будут давать предикт, те для прода использовать модель не стоит. Также нужно учитывать, что она не дала сильного прироста метрики, а даже уступила каким то другим моделям